In [33]:
# To enable horizontal scrolling
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [34]:
%env SPARK_HOME=/opt/spark

env: SPARK_HOME=/opt/spark


# Download of NYC taxi trips and taxi zone file

Modify the base directory in the following cell if you want to save data files in different directories.

In [35]:
base_directory = "./data"

In [36]:
import os
import wget
import zipfile

base_directory = os.path.abspath(base_directory)
os.environ["BASEDIRECTORY"] = base_directory

# Download yellow trip data
data_directory = base_directory + "/taxidata"
data_file = "yellow_tripdata_2022-01.parquet"
data_path = data_directory + "/" + data_file
if not os.path.exists(data_path):
    os.makedirs(data_directory, exist_ok=True)
if not os.path.exists(data_path):
    wget.download("https://d37ci6vzurychx.cloudfront.net/trip-data/" + data_file, out = data_directory)   

# Download zone data
zone_directory = base_directory + "/taxizonesdata"
if not os.path.isdir(zone_directory):
    os.makedirs(zone_directory, exist_ok=True)

zone_zipfile = "taxi_zones.zip"
zone_zipfile_path = zone_directory + "/" + zone_zipfile
if not os.path.exists(zone_zipfile_path):
    wget.download("https://d37ci6vzurychx.cloudfront.net/misc/" + zone_zipfile, out = zone_directory)
    with zipfile.ZipFile(zone_zipfile_path, "r") as zip_ref:
        zip_ref.extractall(zone_directory)
        zip_ref.close()
    
zone_lookup_file = "taxi_zone_lookup.csv"
if not os.path.exists(zone_directory + "/" + zone_lookup_file):
    wget.download("https://d37ci6vzurychx.cloudfront.net/misc/" + zone_lookup_file, out = zone_directory)

# Initialisation of Spark context

In [37]:
import findspark
import os
findspark.init(os.environ['SPARK_HOME'])
print(os.environ['SPARK_HOME'])

/opt/spark


In [38]:
import platform
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Python Spark Map Visualization of NYC taxi trips") \
    .getOrCreate()

sc = spark.sparkContext

In [39]:
trips = spark.read.parquet(data_path)

In [40]:
trips.dtypes

[('VendorID', 'bigint'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'bigint'),
 ('DOLocationID', 'bigint'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double')]

In [41]:
trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-01-01 00:35:40|  2022-01-01 00:53:29|            2.0|          3.8|       1.0|                 N|         142|         236|           1|       14.5|  3.0|    0.5|      3.6

# Grouping using groupBy

In [42]:
# using groupBy
trips.groupBy("VendorID").count().show()

[Stage 28:===================================================>     (9 + 1) / 10]

+--------+-------+
|VendorID|  count|
+--------+-------+
|       6|   5563|
|       5|     36|
|       1| 742273|
|       2|1716059|
+--------+-------+



# Exercise 1
Count trips grouped by passengers.

Are there unexpected values? How can they be interpreted? Find more information on https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page.

In [43]:
trips

DataFrame[VendorID: bigint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: bigint, DOLocationID: bigint, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double]

In [44]:
trips.groupBy("passenger_count").count().orderBy("passenger_count").show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|  71503|
|            0.0|  52061|
|            1.0|1794055|
|            2.0| 343026|
|            3.0|  84570|
|            4.0|  35321|
|            5.0|  51338|
|            6.0|  32037|
|            7.0|      9|
|            8.0|      8|
|            9.0|      3|
+---------------+-------+



# Exercise 2
Find the minimal distance for these groups. Are there unexpected values? How can they be interpreted?

In [45]:
from pyspark.sql.functions import min as spark_min

trips.groupBy("passenger_count") \
     .agg(spark_min("trip_distance").alias("min_distance")) \
     .orderBy("passenger_count") \
     .show()

+---------------+------------+
|passenger_count|min_distance|
+---------------+------------+
|           NULL|         0.0|
|            0.0|         0.0|
|            1.0|         0.0|
|            2.0|         0.0|
|            3.0|         0.0|
|            4.0|         0.0|
|            5.0|         0.0|
|            6.0|         0.0|
|            7.0|         0.0|
|            8.0|         0.0|
|            9.0|         0.0|
+---------------+------------+



# Exercise 3
Remove all trip distances of 0.0 miles from the previous result. What do you expect?

In [46]:
trips.filter(trips.trip_distance > 0.0) \
     .groupBy("passenger_count") \
     .agg(spark_min("trip_distance").alias("min_distance")) \
     .orderBy("passenger_count") \
     .show()

[Stage 37:===================================================>     (9 + 1) / 10]

+---------------+------------+
|passenger_count|min_distance|
+---------------+------------+
|           NULL|        0.01|
|            0.0|        0.01|
|            1.0|        0.01|
|            2.0|        0.01|
|            3.0|        0.01|
|            4.0|        0.01|
|            5.0|        0.01|
|            6.0|        0.01|
|            7.0|        0.32|
|            8.0|        4.09|
|            9.0|        0.12|
+---------------+------------+



# SQL Queries

In [47]:
from pyspark.sql.types import *

sqlContext = SparkSession.builder.getOrCreate()

In [48]:
# Create a temporary view from the DataFrame
trips.createOrReplaceTempView("trips")

In [49]:
# Apply a SQL query
query = "SELECT fare_amount FROM trips WHERE trip_distance>=5"
sqlContext.sql(query).show()

+-----------+
|fare_amount|
+-----------+
|       33.0|
|       17.0|
|       20.0|
|       34.5|
|       21.0|
|       16.0|
|       18.5|
|       52.0|
|       52.0|
|       19.5|
|       23.5|
|       52.0|
|       52.0|
|       24.5|
|       21.5|
|       52.0|
|       19.0|
|       33.0|
|       45.5|
|       20.5|
+-----------+
only showing top 20 rows



# Exercise 4
Rewrite the previous statement without SQL, but with a functional statement.

In [50]:
trips.filter(trips.trip_distance >= 5).select("fare_amount").show()

+-----------+
|fare_amount|
+-----------+
|       33.0|
|       17.0|
|       20.0|
|       34.5|
|       21.0|
|       16.0|
|       18.5|
|       52.0|
|       52.0|
|       19.5|
|       23.5|
|       52.0|
|       52.0|
|       24.5|
|       21.5|
|       52.0|
|       19.0|
|       33.0|
|       45.5|
|       20.5|
+-----------+
only showing top 20 rows



In [51]:
# Compute summary statistics
trips.describe().show()

26/07/06 16:45:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-----------------+--------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|     total_amount|congestion_surcharge|        airport_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-----------------+--------------------+-------

# Exercise 5
Find the distance for tips larger than $5  - Formulate a SQL query and apply it on the DataFrame.

In [52]:
query = "SELECT trip_distance FROM trips WHERE tip_amount > 5"
sqlContext.sql(query).show()

+-------------+
|trip_distance|
+-------------+
|         10.3|
|         5.07|
|         2.48|
|          9.7|
|         6.67|
|         5.03|
|         17.1|
|        19.14|
|         6.49|
|         7.87|
|        18.81|
|          2.7|
|         20.7|
|         11.1|
|        16.75|
|         5.43|
|          5.3|
|        14.48|
|         4.59|
|         4.89|
+-------------+
only showing top 20 rows



# Exercise 6
Formulate a query to get total amount of trip for distances larger than 30 miles.

In [53]:
query = "SELECT total_amount, trip_distance FROM trips WHERE trip_distance > 10 ORDER BY trip_distance DESC"
sqlContext.sql(query).show()

[Stage 49:===================================================>     (9 + 1) / 10]

+------------+-------------+
|total_amount|trip_distance|
+------------+-------------+
|        22.2|    306159.28|
|       18.85|    274658.81|
|       46.72|    250984.47|
|       17.35|    201283.16|
|       45.38|    193150.52|
|        18.0|    180535.93|
|       22.74|    167325.38|
|       41.97|     159571.9|
|       29.05|    153213.84|
|        36.1|     143426.9|
|       36.37|     140442.2|
|        18.5|    123474.27|
|         8.5|    118618.94|
|       14.83|    116986.08|
|       10.21|    112219.77|
|       34.17|    108304.53|
|       20.67|    107994.74|
|       11.61|    107007.93|
|       29.26|    106850.31|
|        15.0|     103676.9|
+------------+-------------+
only showing top 20 rows



# Exercise 7
Create a box-and-whisker plot of the numerical columns. What do these say about the data?

In [54]:
%matplotlib inline

In [55]:
import matplotlib.pyplot as plt

In [ ]:
for column in trips.dtypes:
    name = column[0]
    colType = column[1]
    if colType != 'string' and colType != 'timestamp' and colType != 'timestamp_ntz':
        # approxQuantile(col, [5th, 25th, 50th, 75th, 95th], relativeError=0.0 for exact)
        columnQuantiles = trips.approxQuantile(name, [0.05, 0.25, 0.5, 0.75, 0.95], 0.0)
        print("{} quantiles: {}".format(name, columnQuantiles))
        stats = [{
            "whislo": columnQuantiles[0],  # 5th percentile  – lower whisker
            "q1":     columnQuantiles[1],  # 25th percentile – bottom of box
            "med":    columnQuantiles[2],  # 50th percentile – median line
            "q3":     columnQuantiles[3],  # 75th percentile – top of box
            "whishi": columnQuantiles[4]   # 95th percentile – upper whisker
        }]
        fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(5,5), sharey=True)
        axes.bxp(bxpstats=stats, showfliers=False)
        axes.grid(True)
        axes.set_title(name)

VendorID quantiles: [1.0, 1.0, 2.0, 2.0, 2.0]


passenger_count quantiles: [1.0, 1.0, 1.0, 1.0, 3.0]


trip_distance quantiles: [0.5, 1.04, 1.74, 3.13, 11.8]


RatecodeID quantiles: [1.0, 1.0, 1.0, 1.0, 1.0]


PULocationID quantiles: [48.0, 132.0, 162.0, 234.0, 262.0]


...

mta_tax quantiles: [0.5, 0.5, 0.5, 0.5, 0.5]


tip_amount quantiles: [0.0, 0.72, 2.0, 3.0, 7.16]


tolls_amount quantiles: [0.0, 0.0, 0.0, 0.0, 6.55]


improvement_surcharge quantiles: [0.3, 0.3, 0.3, 0.3, 0.3]


[Stage 76:===================================================>     (9 + 1) / 10]

# Exercise 8
Provide an overview over the number of trips per week day.

In [ ]:
def barchart(dataRows, titleSuffix):
    positions = list(reversed(range(len(dataRows))))
    names = [str(item[titleSuffix]) + " (" + str(item['count']) + ")" for item in dataRows]
    values = [item['count'] for item in dataRows]
    plt.grid()
    plt.barh(positions,values,align="center")
    plt.yticks(positions,names)
    plt.xlabel("Number of trips")
    plt.title("Distribution of trips per " + titleSuffix)
    plt.show()

In [ ]:
import datetime
help(datetime.datetime.weekday)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
import calendar

#udf stands for user defined function
@udf 
def weekdayStr(d):
    return calendar.day_name[d.weekday()]  # returns 'Monday', 'Tuesday', ...

@udf(returnType=IntegerType())
def weekday(d):
    return d.weekday()  # returns 0=Monday ... 6=Sunday

#Replace function weekday with function weekdayStr if you want.
weekdayRows = trips.select(weekday(trips.tpep_dropoff_datetime).alias("weekday")) \
                   .groupBy("weekday") \
                   .count() \
                   .orderBy("weekday") \
                   .collect()

barchart(weekdayRows, "weekday")

# Exercise 9
Provide an overview over the number of trips per hour.

In [ ]:
@udf(returnType=IntegerType())
def hour(d):
    return d.hour  # extracts the hour (0-23) from the datetime

hourRows = trips.select(hour(trips.tpep_dropoff_datetime).alias("hour")) \
                .groupBy("hour") \
                .count() \
                .orderBy("hour") \
                .collect()

barchart(hourRows, "hour")

# Map Visualisations

In [ ]:
import leafmap

In [ ]:
def getMap():
    map_args={
        "google_map":"HYBRID",
        #center to New York at 41 degrees north and 74 degrees west ([lat, lon])
        "center":[40.702557, -74.012318],
        "zoom":12,
        "height":"450px",
        "width":"800px",
        "max_zoom":"20"
    }
    return leafmap.Map(**map_args)

In [ ]:
getMap()

In [ ]:
def taxizoneColorFunction(taxiZonesIntensity, maximum_intensity, taxizoneFeature):
    taxizoneId = taxizoneFeature["properties"]["LocationID"]
    taxizoneIntensity = taxiZonesIntensity[taxizoneId] if taxizoneId in taxiZonesIntensity else 0
    return {
        "color": "black",
        "fillColor": '#%02X0000' % (int(taxizoneIntensity*255/maximum_intensity))
    }
def getTaxiZoneStylingFunction(taxiZonesIntensity):
    maximum_intensity = max(taxiZonesIntensity.values())
    return lambda x: taxizoneColorFunction(taxiZonesIntensity, maximum_intensity, x)

In [ ]:
taxizonesFile = base_directory + "/taxizonesdata/taxi_zones/taxi_zones.shp"
def getZoneCenters():
    zone_centers={}
    my_geojson = leafmap.shp_to_geojson(taxizonesFile)
    for feature in my_geojson["features"]:
        location = feature["properties"]["LocationID"]
        coordinates = feature["geometry"]["coordinates"]
        avg_lat = 0
        avg_lon = 0
        count = 0
        for coordinate_list in coordinates:
            for coordinate in coordinate_list:
                if type(coordinate) == tuple and len(coordinate) == 2:
                    avg_lat += coordinate[1]
                    avg_lon += coordinate[0]
                    count += 1
                elif len(coordinate) > 2:
                    for coord in coordinate:
                        avg_lat += coord[1]
                        avg_lon += coord[0]
                        count += 1
        
        avg_lat = avg_lat/count
        avg_lon = avg_lon/count
        zone_centers[location]=[avg_lat, avg_lon]
    return zone_centers

zoneCenters = getZoneCenters()

In [ ]:
def getHeatCenters(taxizoneIntensityMap):
    heat_data=[]
    for key, value in zoneCenters.items():
        location = key
        (lat, lon) = value
        taxizoneIntensity = taxizoneIntensityMap[location] if location in taxizoneIntensityMap else 0
        heat_data.append([lat, lon, taxizoneIntensity])
    return heat_data

# Exercise 10
Get the number of trips which start/end in each zone.

In [ ]:
pickupData  = trips.groupBy("PULocationID").count().collect()
dropoffData = trips.groupBy("DOLocationID").count().collect()
grouped_by_pickup_location  = {row["PULocationID"]: row["count"] for row in pickupData}
grouped_by_dropoff_location = {row["DOLocationID"]: row["count"] for row in dropoffData}

In [ ]:
m = getMap()
m.add_shp(in_shp=taxizonesFile,layer_name="taxizone",style={},hover_style={}, style_callback=getTaxiZoneStylingFunction(grouped_by_pickup_location), fill_colors=None,
              info_mode='on_hover')
m.layer_opacity('taxizone', 0.9)
m.add_heatmap(data=getHeatCenters(grouped_by_pickup_location), name='pickup_heat', radius=10)
m.layer_opacity('pickup_heat', 0.9)
m

In [ ]:
m = getMap()
m.add_shp(in_shp=taxizonesFile,layer_name="taxizone",style={},hover_style={}, style_callback=getTaxiZoneStylingFunction(grouped_by_dropoff_location), fill_colors=None,
              info_mode='on_hover')
m.layer_opacity('taxizone', 0.9)
m.add_heatmap(data=getHeatCenters(grouped_by_dropoff_location), name='dropoff_heat', radius=10)
m.layer_opacity('dropoff_heat', 0.9)
m

# Exercise 11
Collect the trips with the 10 highest tips. Be careful not to use trips with zones which indicate "Unknown" values.

In [ ]:
zoneLookup = spark.read.csv(base_directory + "/taxizonesdata/taxi_zone_lookup.csv", header=True, inferSchema=True)

In [ ]:
zoneLookup.filter(zoneLookup.Borough == "Unknown").show()
zoneLookup.filter(zoneLookup.Borough == "N/A").show()

In [ ]:
zoneLookup.dtypes

In [ ]:
trips.dtypes

In [ ]:
help(trips.join)

In [ ]:
# Filter out Unknown values, then take the top 10 by tip_amount
# IMPORTANT: use .limit(10) BEFORE .collect() — never collect() on millions of rows!
tripsWithHighestTips = temporary \
    .filter((col("PUBorough") != "Unknown") & (col("DOBorough") != "Unknown")) \
    .limit(10) \
    .collect()
tripsWithHighestTips

In [ ]:
from geojson import FeatureCollection, Feature, LineString
def to_lon_and_lat(latLonCoordinate):
    return [latLonCoordinate[1],latLonCoordinate[0]]

def trip_to_geojson(trip):
    start_point = to_lon_and_lat(zoneCenters[trip["PULocationID"]])
    end_point = to_lon_and_lat(zoneCenters[trip["DOLocationID"]])
    props = {
        "starttime":trip["tpep_pickup_datetime"].isoformat(),
        "startzone":trip["PULocationID"],
        "endtime":trip["tpep_dropoff_datetime"].isoformat(),
        "endzone":trip["DOLocationID"],
    }
    return Feature(geometry=LineString([start_point, end_point]), properties=props)

def tripList_to_geojson(tripList):
    coll = FeatureCollection(list(map(lambda item: trip_to_geojson(item),tripList)))
    return coll

In [ ]:
trip_geojson = tripList_to_geojson(tripsWithHighestTips)

In [ ]:
m = getMap()
m.add_shp(in_shp=taxizonesFile,layer_name="taxizone")
m.layer_opacity('taxizone', 0.9)
m.add_geojson(in_geojson=trip_geojson,layer_name="connections", style={"color":"red"})
m.layer_opacity('connections', 1.0)
m